In [ ]:
from datasets import get_dataset_config_names, get_dataset_split_names

configs = get_dataset_config_names("LeMaterial/LeMat-Synth-Papers")
print("Configs/subsets:", configs)

for c in configs:
    splits = get_dataset_split_names("LeMaterial/LeMat-Synth-Papers", c)
    print(c, "->", splits)

In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "LeMaterial/LeMat-Synth-Papers", "full", split="omg24", streaming=True
)
# print(next(iter(ds)).keys())
for key in next(iter(ds)).keys():
    print(key, "->", type(next(iter(ds))[key]))

In [ ]:
from datasets import get_dataset_config_names, get_dataset_split_names

configs_2 = get_dataset_config_names("LeMaterial/LeMat-Synth")
print("Configs/subsets:", configs_2)

for c in configs_2:
    splits_2 = get_dataset_split_names("LeMaterial/LeMat-Synth", c)
    print(c, "->", splits_2)

In [ ]:
from datasets import load_dataset

ds_2 = load_dataset(
    "LeMaterial/LeMat-Synth-Papers", configs[0], split="full", streaming=True
)
# print(next(iter(ds_2)).keys())
for key in next(iter(ds_2)).keys():
    print(key, "->", type(next(iter(ds_2))[key]))

In [ ]:
import pandas as pd
from datasets import load_dataset

# LeMat-Synth "full" config: one row per material (unfolded from LeMat-Synth-Papers),
# split by source (arxiv/chemrxiv/omg24). evaluation/structured_synthesis are plain dicts.
ds_full = load_dataset("LeMaterial/LeMat-Synth", "full")
df_full = pd.concat(
    [d.to_pandas().assign(source=split) for split, d in ds_full.items()],
    ignore_index=True,
)
print(f"{len(df_full)} materials before filtering")

In [ ]:
def judge_score(row):
    ev = row["evaluation"]
    return ev["scores"]["overall_score"] if ev else None


def num_steps(row):
    synth = row["structured_synthesis"]
    return len(synth["steps"]) if synth and synth.get("steps") else 0


df_full["judge_score"] = df_full.apply(judge_score, axis=1)
df_full["num_steps"] = df_full.apply(num_steps, axis=1)

df_filtered = df_full[(df_full["judge_score"] > 3) & (df_full["num_steps"] > 1)]

print(f"{len(df_filtered)} / {len(df_full)} materials survive filtering")
df_filtered[
    [
        "synthesized_material",
        "material_category",
        "synthesis_method",
        "judge_score",
        "num_steps",
        "source",
    ]
].head()